In [ ]:
import os
import anndata
import scanpy as sc
import pandas as pd
import scanpy as sc
import anndata as ad
import tqdm as notebook_tqdm
from IPython.display import Image, display

pd.set_option('display.max_columns', None)

os.chdir("scRNA-seq")

In [ ]:
adata = sc.read_mtx("data/krizay_2022/matrix.mtx")
adata = adata.T
genes = pd.read_csv("data/krizay_2022/genes.tsv", sep="\t", header=None)
adata.var_names = [line.strip().split("\t")[0] for line in open("data/krizay_2022/genes.tsv")]
adata.obs_names = [line.strip() for line in open("data/krizay_2022/barcodes.tsv")]
adata.var["gene_names"] = genes[1].values

In [23]:
# Filter lowly expressed genes
sc.pp.filter_genes(adata, min_cells = 3)

In [ ]:
adata.layers["counts"] = adata.X.copy() # store the raw counts

# subset count data to highly variable features
sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3", n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]
adata.X = adata.layers["counts"].copy()
adata = adata.copy()


In [ ]:
# run scvi

scvi.model.SCVI.setup_anndata(adata, layer="counts")
model = scvi.model.SCVI(adata)
model.train()
model.save("data/scVI", overwrite=True)

In [ ]:
# extract low dim rep. and save
latent = model.get_latent_representation()
adata.obsm["scVI"] = latent
adata.write('data/scVI/matrix_scvi.h5ad')